### 9.0 Überprüfung und Korrektur der Datentypen
Bei der Überprüfung der Datentypen aller 42 Spalten wurden mehrere Inkonsistenzen
festgestellt, die vor der weiteren Analyse korrigiert wurden.

#### Wetterdaten: `object` statt `float64`
Die sechs Wetterspalten `wind_stability_index`, `weather_temp_mean`,
`weather_temp_trend`, `weather_rain_prob_mean`, `weather_precipitation_mean`
und `weather_humidity_mean` enthielten ausschließlich numerische Werte
(z. B. `26.07`, `0.025`, `59.5`), wurden jedoch von pandas als `object`
eingelesen. Diese Spalten wurden mittels `pd.to_numeric(..., errors="coerce")`
in `float64` konvertiert.

#### Kategorische Textspalten: `object` statt `category`
Die Spalten `meta_race` und `meta_nationality` lagen als generischer
`object`-Typ vor, obwohl sie nur eine begrenzte Anzahl wiederkehrender
Werte enthalten (`meta_race`: 3 eindeutige Werte, `meta_nationality`:
62 eindeutige Werte). Solche Spalten werden als `category`-Typ effizienter
gespeichert und verarbeitet. Beide Spalten wurden entsprechend umgewandelt.

#### Ganzzahlige Punktwerte: `float64` statt `Int64`
Die Spalten `one_day_races`, `gc`, `time_trial`, `sprint`, `climber`,
`hills` sowie `rider_points_season` repräsentieren inhaltlich ganzzahlige
Punktwerte ohne Nachkommastellen. Sie lagen jedoch als `float64` vor. Nach Rundung wurden diese Spalten in den Nullable-Integer-Typ `Int64` konvertiert.

<br>
<br>
Abschließend wurden alle
Nullable-Typen (`Int64`, `Float64`) in die NumPy-Standardtypen (`int64`, `float64`)
konvertiert, um die Kompatibilität mit gängigen Machine-Learning-Bibliotheken
sicherzustellen.



#### Anmerkung:
Die verbleibenden Spalten vom Typ `object`: `meta_url`, `meta_rider_url`,
`meta_url_name`, `meta_name`, `meta_departure`, `meta_arrival`,
`meta_current_team` sowie `meta_url_name` – wurden bewusst nicht konvertiert.
Es handelt sich um Freitext-Bezeichnungen wie Fahrer- und Teamnamen oder
Ortsnamen mit einer hohen Anzahl eindeutiger Werte, für die weder `category`
noch ein anderer spezialisierter Typ einen Mehrwert bieten würde.

In [44]:
import os
os.chdir('/Applications/GADA/GADA-Group3-Cycling-Stage-Prediction/src/Notebooks')

In [45]:
import pickle
import pandas as pd
df = pd.read_pickle('../../data/processed/24_cleaned_master_data.pkl')

### Ausgabe der bisherigen Datentypen:

In [46]:
print(df.dtypes)
with pd.option_context('display.max_columns', None):
    #display(df.describe(include = 'object'))
    display(df.head(5))

meta_race                              object
meta_year                               int64
meta_url                               object
rank                                  float64
meta_rider_url                         object
height                                float64
meta_name                              object
meta_nationality                       object
weight                                  Int64
meta_url_name                          object
meta_departure                         object
meta_arrival                           object
distance                              float64
vertical_meters                         Int64
one_day_races                         float64
gc                                    float64
time_trial                            float64
sprint                                float64
climber                               float64
hills                                 float64
stage_nr                                int64
meta_date                      dat

,meta_race,meta_year,meta_url,rank,meta_rider_url,height,meta_name,meta_nationality,weight,meta_url_name,meta_departure,meta_arrival,distance,vertical_meters,one_day_races,gc,time_trial,sprint,climber,hills,stage_nr,meta_date,meta_departure_lat,meta_departure_lon,meta_arrival_lat,meta_arrival_lon,rider_points_season,rider_rank_season,meta_current_team,team_tier,age_at_race,rider_bmi,race_competitiveness_median,team_power_index,wind_stability_index,weather_temp_mean,weather_temp_trend,weather_rain_prob_mean,weather_precipitation_mean,weather_humidity_mean,won_how_cat,gradient_final_km
0,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,1.0,tom-boonen,1.92,Tom Boonen,BE,82,tom-boonen,Challans,Les Essarts,182.0,432,10498.0,1765.0,1146.0,8626.0,136.0,2296.0,2,2005-07-03,46.847809,-1.877431,46.779115,-1.242859,2149.0,2,Quickstep - Innergetic,elite,24,22.243924,219.0,345.0,0.238106,26.07,4.5,0.025,0.025,59.5,sprint_large_group,0.5
1,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,2.0,thor-hushovd,1.83,Thor Hushovd,NO,83,thor-hushovd,Challans,Les Essarts,182.0,432,4271.0,1237.0,2039.0,5394.0,982.0,3042.0,2,2005-07-03,46.847809,-1.877431,46.779115,-1.242859,1265.0,9,Crédit Agricole,elite,27,24.784258,219.0,119.0,0.238106,26.07,4.5,0.025,0.025,59.5,sprint_large_group,0.5
2,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,3.0,robbie-mcewen,1.71,Robbie McEwen,AU,67,robbie-mcewen,Challans,Les Essarts,182.0,432,4996.0,1461.0,109.0,10554.0,640.0,2650.0,2,2005-07-03,46.847809,-1.877431,46.779115,-1.242859,1444.0,6,Davitamon - Lotto,elite,33,22.913033,219.0,195.0,0.238106,26.07,4.5,0.025,0.025,59.5,sprint_large_group,0.5
3,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,4.0,stuart-o-grady,1.76,Stuart O'Grady,AU,73,stuart-o-grady,Challans,Les Essarts,182.0,432,4053.0,2321.0,2096.0,4146.0,1134.0,2156.0,2,2005-07-03,46.847809,-1.877431,46.779115,-1.242859,977.0,22,"Cofidis, le Crédit par Téléphone",elite,31,23.566632,219.0,206.0,0.238106,26.07,4.5,0.025,0.025,59.5,sprint_large_group,0.5
4,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,5.0,luciano-andre-pagliarini-mendonca,1.74,Luciano André Pagliarini,BR,68,luciano-andre-pagliarini-mendonca,Challans,Les Essarts,182.0,432,259.0,56.0,0.0,637.0,42.0,52.0,2,2005-07-03,46.847809,-1.877431,46.779115,-1.242859,226.0,263,Liquigas,elite,27,22.460034,219.0,223.0,0.238106,26.07,4.5,0.025,0.025,59.5,sprint_large_group,0.5


In [ ]:
# --- Wetter-Spalten: object → float64 ---
weather_cols = [
    "wind_stability_index", "weather_temp_mean", "weather_temp_trend",
    "weather_rain_prob_mean", "weather_precipitation_mean", "weather_humidity_mean"
]
for col in weather_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# --- Kategorische Spalten ---
df["meta_race"] = df["meta_race"].astype("category")
df["meta_nationality"] = df["meta_nationality"].astype("category")

# --- Rider-Punkte-Spalten: float64 → Int64 ---
int_cols = [
    "one_day_races", "gc", "time_trial", "sprint",
    "climber", "hills", "rider_points_season"
]
for col in int_cols:
    df[col] = df[col].round(0).astype("Int64")

# --- Nullable Integer → int64 ---
int64_cols = df.select_dtypes(include="Int64").columns
df[int64_cols] = df[int64_cols].astype("int64")

# --- Nullable Float → float64 ---
float64_cols = df.select_dtypes(include="Float64").columns
df[float64_cols] = df[float64_cols].astype("float64")

# --- NaN-Kontrolle (Wetter-Spalten) ---
print("=== Neue NaN-Werte nach Konvertierung ===")
print(df[weather_cols].isnull().sum())

# --- Finale Datentypen zur Kontrolle ---
print("\n=== Finale Datentypen ===")
print(df.dtypes)

# --- Speichern als neue Pickle-Datei ---
df.to_pickle("../../data/processed/25_cleaned_master_data.pkl")

=== Neue NaN-Werte nach Konvertierung ===
wind_stability_index          0
weather_temp_mean             0
weather_temp_trend            0
weather_rain_prob_mean        0
weather_precipitation_mean    0
weather_humidity_mean         0
dtype: int64

=== Finale Datentypen ===
meta_race                            category
meta_year                               int64
meta_url                               object
rank                                  float64
meta_rider_url                         object
height                                float64
meta_name                              object
meta_nationality                     category
weight                                  int64
meta_url_name                          object
meta_departure                         object
meta_arrival                           object
distance                              float64
vertical_meters                         int64
one_day_races                           int64
gc                                  

In [48]:
df1 = pd.read_pickle("../../data/processed/25_cleaned_master_data.pkl")

print(df1[['meta_current_team', 'meta_departure', 'meta_arrival']])

                       meta_current_team      meta_departure  \
0                 Quickstep - Innergetic            Challans   
1                        Crédit Agricole            Challans   
2                      Davitamon - Lotto            Challans   
3       Cofidis, le Crédit par Téléphone            Challans   
4                               Liquigas            Challans   
...                                  ...                 ...   
196043                    Groupama - FDJ  Robledo de Chavela   
196044       Red Bull - BORA - hansgrohe  Robledo de Chavela   
196045           UAE Team Emirates - XRG  Robledo de Chavela   
196046                           Cofidis  Robledo de Chavela   
196047                           Cofidis  Robledo de Chavela   

                                 meta_arrival  
0                                 Les Essarts  
1                                 Les Essarts  
2                                 Les Essarts  
3                                 Les E